# Notebook 02 — Arquitectura Pix2Pix: U-Net + PatchGAN

> **Blog Técnico** · Fundamentos matemáticos de la arquitectura implementada

Este notebook explica la arquitectura completa de Pix2Pix desde sus fundamentos matemáticos, acompañado de demos de forward pass con tensores reales.

**Contenido:**
1. Generador U-Net 256 — encoder, decoder, skip connections
2. Discriminador PatchGAN 70×70 — campo receptivo
3. Funciones de pérdida — cGAN, LS-GAN, L1
4. Objetivo completo del entrenamiento

---
*Isola et al. (2017) — "Image-to-Image Translation with Conditional Adversarial Networks"*  
https://arxiv.org/abs/1611.07004

In [ ]:
import sys, os
from pathlib import Path
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

PROYECTO = Path('..').resolve() if Path('../src').exists() else Path('.')
os.chdir(PROYECTO)
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

from src.models.generator import GeneradorUNet
from src.models.discriminator import DiscriminadorPatchGAN
from src.models.losses import GANLoss, perdida_total_generador

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

---
## Parte 1 — Generador: U-Net 256

### 1.1 Problema: ¿Por qué no un autoencoder simple?

Un autoencoder convencional codifica la entrada en una representación comprimida (bottleneck) y la decodifica. Para tareas de traducción imagen-a-imagen esto tiene un problema: **la información de estructura de alta frecuencia** (bordes, líneas de carreteras) se pierde durante la compresión y no puede recuperarse fielmente.

### 1.2 La solución: Skip Connections

La U-Net introduce **conexiones de salto** (*skip connections*) que concatenan las activaciones del encoder directamente con las del decoder en el nivel simétrico correspondiente:

$$\mathbf{d}_k = f_\text{dec}\!\left(\left[\mathbf{d}_{k+1}\; ;\; \mathbf{e}_{N-k}\right]\right), \quad k = N{-}1, \ldots, 1$$

donde $[\cdot\;;\;\cdot]$ denota **concatenación de canales**, $\mathbf{e}_k$ son las activaciones del encoder en el nivel $k$, y $N=8$ para imágenes de $256\times256$.

> **¿Por qué concatenación y no suma?** La suma puede cancelar señales cuando $\mathbf{d}_{k+1} \approx -\mathbf{e}_{N-k}$. La concatenación preserva ambas representaciones íntegramente y deja que las convoluciones siguientes decidan cómo combinarlas.

### 1.3 Dimensiones de Canales

Con $n_f = 64$ filtros base (se duplica hasta saturar en 512):

| Nivel | Encoder (entrada→salida) | Resolución | Decoder (entrada→salida) |
|-------|--------------------------|------------|---------------------------|
| 1 — exterior | 3 → 64 | 256→128 | 128 → 3 + Tanh |
| 2 | 64 → 128 | 128→64 | 256 → 64 |
| 3 | 128 → 256 | 64→32 | 512 → 128 |
| 4 | 256 → 512 | 32→16 | 1024 → 256 |
| 5 — dropout | 512 → 512 | 16→8 | 1024 → 512 |
| 6 — dropout | 512 → 512 | 8→4 | 1024 → 512 |
| 7 — dropout | 512 → 512 | 4→2 | 1024 → 512 |
| 8 — bottleneck | 512 → 512 | 2→1 | 512 → 512 (sin cat) |

> El decoder recibe el doble de canales porque el skip connection concatena la salida del subbloque ($\mathbf{d}_{k+1}$) con la activación del encoder correspondiente ($\mathbf{e}_{N-k}$). La única excepción es el bottleneck, donde no hay subbloque y los canales no se duplican.

### 1.4 Dropout en los Tres Bloques Centrales

El paper original aplica `Dropout(0.5)` en los niveles 5, 6 y 7 del decoder. Esto actúa como **regularización** y, crucialmente, introduce **variabilidad en las salidas**: con la misma entrada $x$, el generador puede producir traducciones ligeramente distintas (estocásticas). Sin dropout, la red tiende a mapeos deterministas.

### 1.5 Activación Final: Tanh

$$\hat{y} = \tanh(\mathbf{W}_\text{out} * \mathbf{d}_1) \in [-1, 1]$$

La `Tanh` mapea la salida al rango $[-1, 1]$, compatible con la normalización de la entrada. Usar `Sigmoid` $\in [0,1]$ sería incorrecto aquí porque los datos están normalizados a $[-1,1]$.

In [ ]:
# ── Instanciar el Generador U-Net 256 ─────────────────────────────────────────
G = GeneradorUNet(
    canales_entrada=3,   # imagen RGB de condición
    canales_salida=3,    # imagen RGB generada
    filtros_base=64,     # n_f = 64 (paper original)
).to(device)

n_params = sum(p.numel() for p in G.parameters() if p.requires_grad)
print(f"Generador U-Net 256")
print(f"  Parámetros entrenables : {n_params:,}")
print(f"  ~{n_params/1e6:.1f} M parámetros")

In [ ]:
# ── Forward Pass con hooks para trazar dimensiones ────────────────────────────
registros = {}

def hook_fn(nombre):
    def fn(module, input, output):
        registros[nombre] = output.shape
    return fn

# Registrar encoder y decoder del bloque externo
hooks = [
    G.modelo.encoder.register_forward_hook(hook_fn('enc_externo')),
]

G.eval()
with torch.no_grad():
    x_dummy = torch.randn(1, 3, 256, 256).to(device)
    y_dummy = G(x_dummy)

for h in hooks:
    h.remove()

print("Forward pass completado:")
print(f"  Entrada  x  : {tuple(x_dummy.shape)}  — imagen satelital normalizada")
print(f"  Salida   ŷ  : {tuple(y_dummy.shape)}  — mapa generado")
print(f"  Rango salida: [{y_dummy.min():.3f}, {y_dummy.max():.3f}] — Tanh en [-1,1]")
assert y_dummy.shape == (1, 3, 256, 256), "Forma incorrecta"
print("\n[OK] Generador produce (1, 3, 256, 256) correctamente.")

---
## Parte 2 — Discriminador: PatchGAN 70×70

### 2.1 ¿Por qué no evaluar la imagen completa?

Un discriminador global clasifica toda la imagen de un golpe. Esto tiene dos problemas:
1. Muchos parámetros → más difícil de entrenar
2. Penaliza la imagen en su conjunto → no localiza qué región es falsa

### 2.2 El Campo Receptivo: derivación exacta

El PatchGAN clasifica **parches locales** de tamaño $N\times N$. El tamaño del parche es el **campo receptivo** (RF) de la última capa convolucional.

La fórmula recursiva para el campo receptivo es:

$$RF_n = RF_{n-1} + (k-1) \cdot j_{n-1}, \qquad j_n = j_{n-1} \cdot s_n$$

donde $k=4$ (kernel size fijo), $s_n$ es el stride de la capa $n$, y $j_n$ es el "salto" acumulado (cuántos píxeles de entrada avanza una unidad en el mapa de características).

| Capa | Stride | RF | Jump |
|------|--------|-----|------|
| Entrada | — | **1** | **1** |
| Conv1 (InstanceNorm, LeakyReLU) | 2 | $1 + 3\times1 = $ **4** | **2** |
| Conv2 | 2 | $4 + 3\times2 = $ **10** | **4** |
| Conv3 | 2 | $10 + 3\times4 = $ **22** | **8** |
| Conv4 | 1 | $22 + 3\times8 = $ **46** | **8** |
| Conv5 (salida) | 1 | $46 + 3\times8 = $ **70** | **8** |

Por eso se denomina **PatchGAN 70×70**: cada neurona de la salida cubre un parche de 70×70 píxeles en la imagen de entrada.

### 2.3 Salida del Discriminador

Para una entrada de $256\times256$, la salida es un mapa de $(1, 1, 30, 30)$. Cada una de las $30\times30 = 900$ neuronas produce un escalar que indica si el parche correspondiente es **real** (≈1) o **generado** (≈0).

### 2.4 Entrada de 6 Canales

El discriminador toma **dos imágenes concatenadas**: la imagen de condición $x$ y la imagen objetivo $y$ (real o generada):

$$D(x, y) : \mathbb{R}^{6\times256\times256} \to \mathbb{R}^{1\times30\times30}$$

Esto es esencial en una **cGAN**: el discriminador no juzga si $y$ es realista en general, sino si $y$ es una traducción plausible de $x$. Sin la condición, el discriminador podría aprender a distinguir texturas sin entender la correspondencia semántica.

In [ ]:
# ── Instanciar y probar el Discriminador PatchGAN ─────────────────────────────
D = DiscriminadorPatchGAN(
    canales_entrada=6,   # 3 (condición) + 3 (objetivo) = 6
    filtros_base=64,
).to(device)

n_params_D = sum(p.numel() for p in D.parameters() if p.requires_grad)
print(f"Discriminador PatchGAN 70×70")
print(f"  Parámetros: {n_params_D:,}")
print(f"  RF = 70 px (ver tabla en Markdown)")

# Forward pass: D(condición, objetivo)
condicion = torch.randn(1, 3, 256, 256).to(device)  # imagen A (satelital)
objetivo  = torch.randn(1, 3, 256, 256).to(device)  # imagen B (mapa)

D.eval()
with torch.no_grad():
    pred = D(condicion, objetivo)

print(f"\nForward D(x, y):")
print(f"  x (condición) : {tuple(condicion.shape)}")
print(f"  y (objetivo)  : {tuple(objetivo.shape)}")
print(f"  D(x,y)        : {tuple(pred.shape)}  ← mapa 30×30 de realismo por parche")
print(f"  Rango salida  : [{pred.min():.3f}, {pred.max():.3f}]")
assert pred.shape == (1, 1, 30, 30), f"Forma incorrecta: {pred.shape}"
print("\n[OK] Discriminador produce (1, 1, 30, 30) correctamente.")

---
## Parte 3 — Funciones de Pérdida

### 3.1 cGAN Clásica (BCE)

El objetivo de una GAN condicional ($x$ es la imagen de condición, $z$ es ruido, $y$ es el objetivo real):

$$\mathcal{L}_\text{cGAN}(G,D) = \mathbb{E}_{x,y}\bigl[\log D(x,y)\bigr] + \mathbb{E}_{x,z}\bigl[\log\bigl(1 - D(x,G(x,z))\bigr)\bigr]$$

- **D** maximiza esta expresión: aprende a distinguir pares reales de falsos.
- **G** minimiza el segundo término: aprende a generar imágenes que D clasifique como reales.

### 3.2 LS-GAN (MSE) — Variante usada en este proyecto

La cGAN clásica con BCE tiene problemas de **vanishing gradient**: cuando el discriminador es muy bueno, $\log(1-D(x,G(x,z))) \approx 0$ y los gradientes hacia G se extinguen.

La solución de **LS-GAN** (Mao et al., 2017) usa MSE:

$$\mathcal{L}_D^\text{LS} = \frac{1}{2}\mathbb{E}_{x,y}\bigl[(D(x,y) - 1)^2\bigr] + \frac{1}{2}\mathbb{E}_{x,z}\bigl[D(x,G(x,z))^2\bigr]$$

$$\mathcal{L}_G^\text{LS} = \frac{1}{2}\mathbb{E}_{x,z}\bigl[(D(x,G(x,z)) - 1)^2\bigr]$$

Esto penaliza muestras reales **lejanas a 1** y muestras falsas **lejanas a 0**, con gradientes no nulos incluso cuando D es preciso.

### 3.3 Pérdida L1 — Fuerza Correspondencia Pixel a Pixel

$$\mathcal{L}_{L1}(G) = \mathbb{E}_{x,y,z}\bigl[\|y - G(x,z)\|_1\bigr]$$

> **¿Por qué L1 y no L2?**
> El mínimo de L2 (MSE) es la **media** sobre todas las posibles salidas → imágenes borrosas.
> El mínimo de L1 es la **mediana** → imágenes más nítidas con mejores bordes.

### 3.4 Objetivo Completo

$$G^* = \arg\min_G \max_D \; \mathcal{L}_\text{cGAN}(G, D) + \lambda \cdot \mathcal{L}_{L1}(G), \quad \lambda = 100$$

**¿Por qué $\lambda = 100$?** Con pesos aleatorios iniciales:
- $\mathcal{L}_\text{GAN} \approx 0.65$ (entropía cruzada con predicción ≈ 0.5)
- $\mathcal{L}_{L1} \approx 0.24$ (píxeles independientes en $[-1,1]$)

Por tanto $\lambda = 100$ da a L1 un peso relativo de $\frac{100 \times 0.24}{0.65 + 100 \times 0.24} \approx 97\%$, asegurando que el generador priorice la **estructura global** (L1) sobre los detalles de textura (GAN).

In [ ]:
# ── Demo de las funciones de pérdida ──────────────────────────────────────────
criterio_gan = GANLoss(gan_mode='lsgan').to(device)
criterio_l1  = nn.L1Loss()

G.train()
D.train()

x_cond = torch.randn(1, 3, 256, 256).to(device)  # imagen de condición
y_real = torch.randn(1, 3, 256, 256).to(device)  # imagen objetivo real

# ── Paso del Discriminador ────────────────────────────────────────────────────
# G genera la imagen falsa
with torch.no_grad():
    y_fake = G(x_cond)

# D evalúa pares real y falso
pred_real = D(x_cond, y_real)
pred_fake = D(x_cond, y_fake.detach())  # .detach(): no propagar gradientes a G aquí

loss_D_real = criterio_gan(pred_real, es_real=True)
loss_D_fake = criterio_gan(pred_fake, es_real=False)
loss_D = (loss_D_real + loss_D_fake) * 0.5

# ── Paso del Generador ────────────────────────────────────────────────────────
# G intenta engañar a D (pide que clasifique su salida como real)
y_fake = G(x_cond)
pred_fake_G = D(x_cond, y_fake)  # sin .detach(): gradientes fluyen hacia G

loss_G_GAN = criterio_gan(pred_fake_G, es_real=True)
loss_G_L1  = criterio_l1(y_fake, y_real)
loss_G     = perdida_total_generador(loss_G_GAN, loss_G_L1, lambda_l1=100.0)

print("Pérdidas en paso inicial (pesos aleatorios):")
print(f"  loss_D_real = {loss_D_real.item():.4f}  (LS-GAN: (D(x,y)-1)²/2)")
print(f"  loss_D_fake = {loss_D_fake.item():.4f}  (LS-GAN: D(x,G(x))²/2)")
print(f"  loss_D      = {loss_D.item():.4f}  = (real + fake) / 2")
print()
print(f"  loss_G_GAN  = {loss_G_GAN.item():.4f}  (LS-GAN: (D(x,G(x))-1)²/2)")
print(f"  loss_G_L1   = {loss_G_L1.item():.4f}  (L1 media por píxel)")
print(f"  loss_G      = {loss_G.item():.4f}  = GAN + 100 × L1")
print()
print(f"  Contribución relativa de L1: {100 * loss_G_L1.item() / loss_G.item() * 100:.1f}%")

---
## Parte 4 — Visualización de la Arquitectura

Diagrama simplificado del flujo de datos en un paso de entrenamiento Pix2Pix:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Arquitectura Pix2Pix: Flujo de Entrenamiento', fontsize=13)

# ── Diagrama izquierdo: U-Net ──────────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Generador U-Net 256', fontsize=11)

# Encoder
capas_enc = [
    (0.5, 9.0, '3→64\n256²',   'steelblue'),
    (0.5, 7.5, '64→128\n128²', 'steelblue'),
    (0.5, 6.0, '128→256\n64²', 'steelblue'),
    (0.5, 4.5, '256→512\n32²', 'steelblue'),
    (0.5, 3.0, '512→512\nbottleneck', 'navy'),
]
for x, y, txt, col in capas_enc:
    ax.add_patch(plt.Rectangle((x, y-0.55), 2.5, 1.0, color=col, alpha=0.7))
    ax.text(x+1.25, y, txt, ha='center', va='center', color='white', fontsize=7)

# Decoder
capas_dec = [
    (7.0, 9.0, '128→3\nTanh',    'darkorange'),
    (7.0, 7.5, '256→64',         'darkorange'),
    (7.0, 6.0, '512→128',        'darkorange'),
    (7.0, 4.5, '1024→256',       'darkorange'),
    (7.0, 3.0, '1024→512\ndropout', 'chocolate'),
]
for x, y, txt, col in capas_dec:
    ax.add_patch(plt.Rectangle((x, y-0.55), 2.5, 1.0, color=col, alpha=0.7))
    ax.text(x+1.25, y, txt, ha='center', va='center', color='white', fontsize=7)

# Flechas skip connections
for y in [9.0, 7.5, 6.0, 4.5]:
    ax.annotate('', xy=(7.0, y), xytext=(3.0, y),
                arrowprops=dict(arrowstyle='->', color='green', lw=1.5,
                                connectionstyle='arc3,rad=-0.2'))

ax.text(5.0, 9.5, 'skip\nconnections', ha='center', color='green', fontsize=8)

# Labels
enc_patch = mpatches.Patch(color='steelblue', label='Encoder (LeakyReLU + Conv + IN)')
dec_patch = mpatches.Patch(color='darkorange', label='Decoder (ReLU + ConvT + IN)')
skip_patch = mpatches.Patch(color='green', label='Skip connection (cat)')
ax.legend(handles=[enc_patch, dec_patch, skip_patch], loc='lower center',
          fontsize=7, ncol=1)

# ── Diagrama derecho: PatchGAN ────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('Discriminador PatchGAN 70×70', fontsize=11)

capas_d = [
    (1, 8.5, '6 ch\n256²',   'purple'),
    (1, 7.0, '64 ch\n128²',  'mediumpurple'),
    (1, 5.5, '128 ch\n64²',  'mediumpurple'),
    (1, 4.0, '256 ch\n32²',  'mediumpurple'),
    (1, 2.5, '512 ch\n31²',  'mediumpurple'),
    (1, 1.0, '1 ch\n30²',    'darkviolet'),
]
for x, y, txt, col in capas_d:
    ax2.add_patch(plt.Rectangle((x, y-0.55), 2.5, 1.0, color=col, alpha=0.75))
    ax2.text(x+1.25, y, txt, ha='center', va='center', color='white', fontsize=8)

# RF annotation
for i, (rf, j) in enumerate([(4,2),(10,4),(22,8),(46,8),(70,8)]):
    ax2.text(6.5, 8.5 - i*1.5,
             f'RF={rf}px  j={j}',
             fontsize=8, color='darkviolet')

ax2.text(4.5, 0.1, 'Salida: (1,1,30,30) — realismo por parche',
         ha='center', fontsize=8, color='darkviolet')

plt.tight_layout()
plt.savefig('results/arquitectura_pix2pix.png', dpi=130, bbox_inches='tight')
plt.show()
print("Diagrama guardado en results/arquitectura_pix2pix.png")

---
## Resumen de la Arquitectura

| Componente | Parámetros | Entrada | Salida |
|---|---|---|---|
| Generador U-Net 256 | ~54 M | $(1,3,256,256)$ | $(1,3,256,256)$ |
| Discriminador PatchGAN | ~2.8 M | $(1,6,256,256)$ | $(1,1,30,30)$ |

**Ratio G/D de parámetros ≈ 19:1** — es intencionado: el generador debe aprender una transformación compleja (imagen completa) mientras el discriminador solo juzga parches locales.

**Por qué funciona LS-GAN + L1:**
- La **L1** ($\lambda=100$) actúa como un prior fuerte: la imagen generada debe parecerse estructuralmente al objetivo.
- La **LS-GAN** añade los detalles de textura y realismo que la L1 sola no puede capturar (tendería a producir imágenes borrosas).

Siguiente notebook: **03_training_sat2sketch.ipynb**